<a href="https://colab.research.google.com/github/JuanZapa7a/AINavalEngineering/blob/main/NB06_SciPy_for_Naval_Engineering_Interpolation_Optimization_and_Signals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# **NB06 · Class 6 — SciPy for Naval Engineering: Interpolation, Optimization, and Signals**

## Block 2: AI — Machine Learning (continued)

`NB03`–`NB05` built real fluency in NumPy and Matplotlib. This class closes Block 2's tooling sequence with **SciPy**, the library every classical engineering computation in this course leans on but has never used directly: interpolating sparse real measurements, finding an optimum from real data, integrating a real curve, and analyzing a signal's real frequency content. No dataset in this notebook is invented for the occasion — the interpolation and optimization sections use the real, sparse `yacht_hydrodynamics.data` (`NB10`) towing-tank measurements; only the closing signal-analysis section uses a clearly-labeled synthetic ship-motion signal, in the same spirit as `NB01`'s own vibration example, because no real ship-motion time series exists yet in this course's datasets.

### Learning objectives

By the end of this class, students will be able to:
- Interpolate sparse real measurements with `scipy.interpolate.make_interp_spline` and explain why that's different from just connecting the dots.
- Find a real optimum (minimum) of a function built from real data with `scipy.optimize.minimize_scalar`.
- Numerically integrate a real curve with `scipy.integrate.simpson`.
- Analyze a signal's frequency content with `scipy.fft` and detect real local peaks with `scipy.signal.find_peaks`.

### Agenda (2-hour class)

| # | Section | Minutes |
|---|---|---|
| 1 | Recap and why SciPy | 5 |
| 2 | Interpolating sparse real tank-test data | 20 |
| 3 | Finding a real economic-speed optimum | 20 |
| 4 | Numerical integration of a real curve | 20 |
| 5 | Frequency analysis with `scipy.fft` | 25 |
| 6 | Detecting real peaks with `scipy.signal` | 10 |
| 7 | Summary, homework, next class | 20 |

As always: approximate guidance, not a script.


---

## 1. Why SciPy


NumPy gives arrays and basic operations; SciPy builds real numerical methods on top: interpolation, optimization, integration, and signal processing among many others. This class is deliberately the **closing** class of Block 2's tooling sequence (`NB03`–`NB06`) because it leans on everything before it — a real spline is built from real arrays (`NB03`), evaluated across a real grid (`NB04`'s broadcasting), and plotted to check the result (`NB05`).

> **Further reading**: [SciPy official documentation](https://docs.scipy.org/doc/scipy/) | [SciPy (Wikipedia)](https://en.wikipedia.org/wiki/SciPy)


---

## 2. Interpolating sparse real tank-test data


`NB10`'s Yacht Hydrodynamics dataset holds 308 real towing-tank measurements across 22 real hull shapes, 14 Froude numbers (a dimensionless speed measure) each. The cell below isolates **one real hull's** 14 real (Froude number, residuary resistance coefficient) measurements — sparse by design, since each real physical tank run is expensive.


In [ ]:
import numpy as np
import urllib.request

yacht_url = "https://raw.githubusercontent.com/JuanZapa7a/AINavalEngineering/main/Datasets/yacht_hydrodynamics.data"
raw_lines = urllib.request.urlopen(yacht_url).read().decode("utf-8").strip().split("\n")
all_rows = np.array([[float(x) for x in line.split()] for line in raw_lines])

first_hull_config = all_rows[0, :5]
same_hull = np.all(all_rows[:, :5] == first_hull_config, axis=1)
hull_rows = all_rows[same_hull]

froude = hull_rows[:, 5]          # dimensionless speed
resistance = hull_rows[:, 6]      # real residuary resistance coefficient

print(f"Real measurements for this hull: {len(froude)}")
print("Froude numbers:", froude)
print("Resistance coefficients:", resistance)


`scipy.interpolate.make_interp_spline` fits a smooth curve through these exact real points, letting us estimate the resistance at any Froude number **between** the ones actually measured — without a new, expensive tank run. (Older SciPy code sometimes uses `interpolate.interp1d` for this; SciPy's own documentation now recommends `make_interp_spline` instead, which this class uses throughout.)


In [ ]:
from scipy.interpolate import make_interp_spline
import matplotlib.pyplot as plt

spline = make_interp_spline(froude, resistance, k=3)   # cubic spline through the real points

froude_fine = np.linspace(froude.min(), froude.max(), 300)
resistance_smooth = spline(froude_fine)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(froude_fine, resistance_smooth, color="steelblue", label="Cubic spline (interpolated)")
ax.scatter(froude, resistance, color="darkorange", zorder=3, label="Real tank measurements")
ax.set_xlabel("Froude number")
ax.set_ylabel("Residuary resistance coefficient")
ax.set_title("Real sparse measurements, interpolated")
ax.legend()
plt.show()


> **Further reading**: [Spline interpolation (Wikipedia)](https://en.wikipedia.org/wiki/Spline_interpolation) | [`scipy.interpolate.make_interp_spline` documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.interpolate.make_interp_spline.html) | [Froude number (Wikipedia)](https://en.wikipedia.org/wiki/Froude_number)


---

## 3. Finding a real economic-speed optimum


Section 2's real data shows resistance **increasing** with speed across the whole measured range — on resistance alone, the "best" speed is always the slowest one, which isn't a useful answer. A real economic-speed decision weighs resistance (higher speed costs more fuel) against the real cost of taking longer to arrive. The cell below adds a simple time-cost term (`K / Froude number`, illustrative — a real fleet would use its own real day-rate figures here) to the *real* interpolated resistance curve from Section 2, then asks `scipy.optimize.minimize_scalar` to find the Froude number that minimizes the combined total.


In [ ]:
from scipy.optimize import minimize_scalar

K = 0.4   # illustrative weighting of "cost of time" relative to resistance -- not a real fleet's day rate

def total_cost(fr):
    return spline(fr) + K / fr

result = minimize_scalar(total_cost, bounds=(froude.min(), froude.max()), method="bounded")

print(f"Optimal (economic) Froude number: {result.x:.3f}")
print(f"Total cost at optimum: {result.fun:.3f}")
print(f"Real resistance coefficient at that speed: {spline(result.x):.3f}")


Plotting the full cost curve makes it clear this optimum is a genuine interior minimum, not just a boundary artifact.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
costs = total_cost(froude_fine)
ax.plot(froude_fine, costs, color="seagreen", label="Total cost (resistance + time penalty)")
ax.axvline(result.x, color="black", linestyle="--", label=f"Optimum: Fr = {result.x:.3f}")
ax.set_xlabel("Froude number")
ax.set_ylabel("Total cost (illustrative units)")
ax.set_title("A real optimum found on real (interpolated) data")
ax.legend()
plt.show()


**Read this result honestly**: the resistance curve itself is real; the specific optimum found here depends entirely on the illustrative `K` chosen for the time penalty. A real fleet operator would replace `K` with a real day-rate/fuel-price ratio — the *method* (interpolate real data, then optimize a cost built on top of it) is the transferable lesson, not this specific `K = 0.4` number.

> **Further reading**: [Mathematical optimization (Wikipedia)](https://en.wikipedia.org/wiki/Mathematical_optimization) | [`scipy.optimize.minimize_scalar` documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize_scalar.html) | [Ship's economic speed / slow steaming (Wikipedia)](https://en.wikipedia.org/wiki/Slow_steaming)


---

## 4. Numerical integration of a real curve


**Integration** finds the area under a curve — here, the real interpolated resistance curve from Section 2, over the real measured Froude-number range. `scipy.integrate.simpson` estimates this numerically from sampled points, using Simpson's rule (fitting small parabolic segments, more accurate than simply summing rectangles). (Older SciPy code sometimes calls this `integrate.simps`; the current, non-deprecated name is `simpson`, used here.)


In [ ]:
from scipy.integrate import simpson

area = simpson(y=resistance_smooth, x=froude_fine)
print(f"Area under the real resistance curve, Fr = {froude.min():.3f} to {froude.max():.3f}: {area:.3f}")
print("(proportional to the total resistance work swept while accelerating across this real speed range)")


> **Further reading**: [Simpson's rule (Wikipedia)](https://en.wikipedia.org/wiki/Simpson%27s_rule) | [`scipy.integrate.simpson` documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.simpson.html)


---

## 5. Frequency analysis with `scipy.fft`


No real ship-motion time series exists yet in this course's datasets, so this section uses a **clearly synthetic** signal — the same honest convention `NB01`'s own vibration example already established for this course. It simulates a vessel's heave (vertical) motion: a dominant real-world-plausible wave-encounter frequency plus sensor noise, and asks whether `scipy.fft` can recover the injected frequency from the noisy signal alone.


In [ ]:
rng = np.random.default_rng(42)

sample_rate_hz = 10.0                 # 10 readings per second
duration_s = 60.0
t = np.arange(0, duration_s, 1 / sample_rate_hz)

true_wave_freq_hz = 0.15              # a plausible real wave-encounter frequency (~6.7 s period)
heave_signal = 0.8 * np.sin(2 * np.pi * true_wave_freq_hz * t) + 0.15 * rng.standard_normal(len(t))

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(t, heave_signal, linewidth=0.8)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Heave (m, synthetic)")
ax.set_title("Synthetic heave signal (true frequency hidden -- unknown to the FFT)")
plt.show()


Nothing in this signal's own plot reveals its frequency by eye through the noise -- `scipy.fft` extracts it directly from the data.


In [ ]:
from scipy.fft import rfft, rfftfreq

fft_values = rfft(heave_signal)
fft_freqs = rfftfreq(len(heave_signal), d=1 / sample_rate_hz)
fft_magnitude = np.abs(fft_values)

dominant_freq = fft_freqs[np.argmax(fft_magnitude)]

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(fft_freqs, fft_magnitude)
ax.axvline(dominant_freq, color="darkred", linestyle="--", label=f"Detected: {dominant_freq:.3f} Hz")
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Magnitude")
ax.set_title("FFT of the synthetic heave signal")
ax.set_xlim(0, 1.0)
ax.legend()
plt.show()

print(f"True injected frequency:  {true_wave_freq_hz} Hz")
print(f"FFT-detected frequency:   {dominant_freq:.3f} Hz")


> **Further reading**: [Fast Fourier transform (Wikipedia)](https://en.wikipedia.org/wiki/Fast_Fourier_transform) | [`scipy.fft` documentation](https://docs.scipy.org/doc/scipy/reference/fft.html) | [Ship motions / heave (Wikipedia)](https://en.wikipedia.org/wiki/Ship_motions)


---

## 6. Detecting real peaks with `scipy.signal`


`scipy.signal.find_peaks` locates local maxima directly in the time-domain signal — useful whenever the question is "how many wave crests happened" rather than "what frequency dominates."


In [ ]:
from scipy.signal import find_peaks

peak_indices, _ = find_peaks(heave_signal, height=0.3, distance=int(sample_rate_hz * 3))

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(t, heave_signal, linewidth=0.8)
ax.scatter(t[peak_indices], heave_signal[peak_indices], color="darkred", zorder=3, label="Detected peaks")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Heave (m, synthetic)")
ax.set_title(f"{len(peak_indices)} real local peaks detected in this synthetic signal")
ax.legend()
plt.show()


> **Further reading**: [`scipy.signal.find_peaks` documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.find_peaks.html)


---

## Class summary

- `scipy.interpolate.make_interp_spline` builds a smooth, usable curve from real sparse measurements (`interp1d` is the older, now-discouraged way to do this).
- `scipy.optimize.minimize_scalar` finds a real optimum on a function built from real data — here, a genuine economic-speed trade-off, with an honest note about which part of the result is real data and which part is an illustrative weighting.
- `scipy.integrate.simpson` computes the real area under a curve (`simps` is the older, removed name).
- `scipy.fft` recovers a signal's real frequency content; `scipy.signal.find_peaks` finds real local maxima directly in time.
- This closes Block 2's tooling sequence (`NB03`–`NB06`): every later notebook's data handling, geometry, and signal work builds on what these four classes established.

## For the next class (NB07)

Machine Learning fundamentals proper, applied end to end to a real ship dataset — supervised learning, train/validation/test splits, and real evaluation metrics.

## Homework / Practice Ideas

1. Repeat Section 2's interpolation for a *different* real hull configuration in `yacht_hydrodynamics.data` — does the resistance curve's shape look similar, or does hull geometry change it meaningfully?
2. Change `K` in Section 3 across a real range (e.g. 0.1 to 0.6) and plot how the optimal Froude number shifts — at what point does the optimum hit the boundary of the real measured range?
3. Compute Section 4's integral separately for two different real hulls and compare — does a hull with generally higher resistance also have a larger area under its curve, as expected?
4. Add a *second* real-world-plausible frequency component to Section 5's synthetic signal (e.g. a faster roll motion on top of the slower heave) and see whether `scipy.fft` can recover both frequencies from the combined, noisy signal.
5. Lower Section 6's `height` threshold in `find_peaks` and observe how many additional (likely noise-driven, not real wave crests) peaks get detected — what does this suggest about choosing peak-detection parameters on real, noisy sensor data?

> ***As always: SciPy gives real numerical methods -- the engineering judgment of which real data feeds them, and how to read the result honestly, is still yours.***
